In [6]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error
import joblib

# 加载数据
file_path = "/home/zhanyu/experiment/data-hh/my/train_86i3lia5Jj4=_aggregated_output.csv"
data = pd.read_csv(file_path)

# 数据预处理
data['flt_date'] = pd.to_datetime(data['flt_date'])
data['flt_date'] = (data['flt_date'] - data['flt_date'].min()).dt.days

label_fields = ['flt_no', 'a', 'b', 'c', 'aircraft']
label_encoders = {col: LabelEncoder() for col in label_fields}
for col in label_fields:
    data[col] = label_encoders[col].fit_transform(data[col])

X = data[['flt_date', 'flt_no', 'a', 'b', 'c', 'aircraft', 'ab_duration', 'bc_duration', 'ac_duration']]
y = data[['ab_pax', 'bc_pax', 'ac_pax']]

# 分割数据
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 训练模型
ab_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
bc_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
ac_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)

ab_model.fit(X_train, y_train['ab_pax'])
bc_model.fit(X_train, y_train['bc_pax'])
ac_model.fit(X_train, y_train['ac_pax'])

# 评估模型
ab_preds = ab_model.predict(X_test)
bc_preds = bc_model.predict(X_test)
ac_preds = ac_model.predict(X_test)

ab_mse = mean_squared_error(y_test['ab_pax'], ab_preds)
bc_mse = mean_squared_error(y_test['bc_pax'], bc_preds)
ac_mse = mean_squared_error(y_test['ac_pax'], ac_preds)

print(f"AB_PAX MSE: {ab_mse}")
print(f"BC_PAX MSE: {bc_mse}")
print(f"AC_PAX MSE: {ac_mse}")

# # 保存模型
# joblib.dump(ab_model, "ab_model.pkl")
# joblib.dump(bc_model, "bc_model.pkl")
# joblib.dump(ac_model, "ac_model.pkl")

# 获取测试集前 5 项数据
X_test_sample = X_test.head(5)
y_test_sample = y_test.head(5)

# 获取预测结果
ab_preds_sample = ab_model.predict(X_test_sample)
bc_preds_sample = bc_model.predict(X_test_sample)
ac_preds_sample = ac_model.predict(X_test_sample)

# 创建结果 DataFrame
results = pd.DataFrame({
    'AB_PAX_True': y_test_sample['ab_pax'].values,
    'AB_PAX_Pred': ab_preds_sample,
    'BC_PAX_True': y_test_sample['bc_pax'].values,
    'BC_PAX_Pred': bc_preds_sample,
    'AC_PAX_True': y_test_sample['ac_pax'].values,
    'AC_PAX_Pred': ac_preds_sample,
})

print("前 5 项预测结果与真实值对比：")
print(results)

AB_PAX MSE: 416.731814541593
BC_PAX MSE: 345.0380245510838
AC_PAX MSE: 217.2253020855363
前 5 项预测结果与真实值对比：
   AB_PAX_True  AB_PAX_Pred  BC_PAX_True  BC_PAX_Pred  AC_PAX_True  \
0          104    91.814705           88    83.802406           38   
1           93    75.078194          107    99.153465           39   
2           99    75.695091           69    81.976700           57   
3           89    99.823669           86   102.657951           60   
4           47    51.437202           53    90.534248           37   

   AC_PAX_Pred  
0    27.975676  
1    50.101665  
2    67.083054  
3    49.086308  
4    46.501823  


前 5 项预测结果与真实值对比：
   AB_PAX_True  AB_PAX_Pred  BC_PAX_True  BC_PAX_Pred  AC_PAX_True  \
0          104    91.814705           88    83.802406           38   
1           93    75.078194          107    99.153465           39   
2           99    75.695091           69    81.976700           57   
3           89    99.823669           86   102.657951           60   
4           47    51.437202           53    90.534248           37   

   AC_PAX_Pred  
0    27.975676  
1    50.101665  
2    67.083054  
3    49.086308  
4    46.501823  
